In [ ]:
import wrds
import pandas as pd
import os
from sqlalchemy import text

In [ ]:
START = "2011-01-01"
END   = "2025-12-31"

TICKERS = {
    "AAPL":  "AAPL",
    "MSFT":  "MSFT",
    "NVDA":  "NVDA",   # Nvidia (replacing Google — full history from 2011)
    "AMZN":  "AMZN",
    "JPM":   "JPM",
    "JNJ":   "JNJ",
    "XOM":   "XOM",
    "TSLA":  "TSLA",
    "NFLX":  "NFLX",
    "V":     "V",
}
ticker_list = tuple(TICKERS.values())

In [43]:
# WRDS usernames are lowercase — pass explicitly to avoid system username being used
# First-time login will prompt for your password and cache it in ~/.pgpass
db = wrds.Connection()  # replace with your WRDS username if different

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [ ]:
query = f"""
    SELECT b.ticker, a.date,
        ABS(a.prc) AS close, a.openprc AS open,
        a.askhi AS high, a.bidlo AS low,
        a.vol AS volume, a.ret AS return
    FROM crsp.dsf AS a
    JOIN crsp.dsenames AS b
        ON  a.permno = b.permno
        AND a.date  >= b.namedt
        AND a.date  <= COALESCE(b.nameendt, CURRENT_DATE)
    WHERE b.ticker IN {ticker_list}
      AND a.date BETWEEN '{START}' AND '{END}'
    ORDER BY b.ticker, a.date
"""

with db.engine.connect() as conn:
    df_raw = pd.read_sql(text(query), conn, parse_dates=["date"])

print(f"Downloaded {len(df_raw):,} rows across {df_raw['ticker'].nunique()} tickers.")
df_raw.head()

In [ ]:
# Check which tickers actually came back
print("Tickers in result:", sorted(df_raw["ticker"].unique()))

# Split into per-ticker DataFrames
raw = {}
for name, crsp_ticker in TICKERS.items():
    sub = df_raw[df_raw["ticker"] == crsp_ticker].copy()
    if sub.empty:
        print(f"WARNING: {name} ({crsp_ticker}) returned no data — check ticker spelling in CRSP")
        continue
    sub = sub.set_index("date").drop(columns="ticker")
    sub.index = pd.to_datetime(sub.index)
    raw[name] = sub
    print(f"{name:6s}: {len(sub):4d} rows  |  {sub.index[0].date()} -> {sub.index[-1].date()}  |  NAs: {sub.isna().sum().sum()}")

In [ ]:
# Summary statistics on close prices
for name, df in raw.items():
    print(f"\n--- {name} ---")
    display(df["close"].describe())

In [ ]:
os.makedirs("../data/stocks", exist_ok=True)

for name, df in raw.items():
    out = f"../data/stocks/raw_{name.lower()}.csv"
    df.to_csv(out)
    print(f"Saved {out}")

db.close()